In [ ]:
# If needed, uncomment:
# !pip install -q pandas numpy matplotlib seaborn

import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# ================================================================
# 1. LOAD DATA
# ================================================================
file_path = "Table1.csv"

try:
    df = pd.read_csv(file_path, low_memory=False)
    print(f"File loaded successfully: {file_path}")
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
except FileNotFoundError:
    print(f"Error: file {file_path} was not found.")
    raise


# ================================================================
# 2. VARIABLE ENGINEERING (TEXT MINING + CHECKBOXES) WITH NAN PROTECTION
# ================================================================
text_cols = [c for c in df.columns if df[c].dtype == object]
df_text = df[text_cols].fillna("")

regex_htn = r"\b(hipertens[aã]o|has|press[aã]o alta)\b"
regex_dm = r"\b(diabetes|dm2|dm1|dm)\b"
regex_dlp = r"\b(colesterol|dislipidemia|dlp|triglic[eé]rides)\b"

cond_htn_text = df_text.apply(
    lambda x: x.astype(str).str.contains(regex_htn, case=False, regex=True)
).any(axis=1)
cond_dm_text = df_text.apply(
    lambda x: x.astype(str).str.contains(regex_dm, case=False, regex=True)
).any(axis=1)
cond_dlp_text = df_text.apply(
    lambda x: x.astype(str).str.contains(regex_dlp, case=False, regex=True)
).any(axis=1)

chronic_cols = [
    c for c in df.columns
    if "doenças crônicas" in c.lower() and "choice=" in c.lower()
]

cond_htn_struct = pd.Series(False, index=df.index)
cond_dm_struct = pd.Series(False, index=df.index)
cond_dlp_struct = pd.Series(False, index=df.index)

if chronic_cols:
    for c in chronic_cols:
        is_checked = df[c].astype(str).str.lower().str.strip().isin(
            ["checked", "1", "1.0", "sim", "verdadeiro"]
        )

        if "hipertensão" in c.lower() or "has" in c.lower():
            cond_htn_struct = cond_htn_struct | is_checked
        if "diabetes" in c.lower() or "dm" in c.lower():
            cond_dm_struct = cond_dm_struct | is_checked
        if "colesterol" in c.lower() or "dislipidemia" in c.lower():
            cond_dlp_struct = cond_dlp_struct | is_checked

df["Has_Hypertension"] = (cond_htn_text | cond_htn_struct).astype(int)
df["Has_Diabetes"] = (cond_dm_text | cond_dm_struct).astype(int)
df["Has_Dyslipidemia"] = (cond_dlp_text | cond_dlp_struct).astype(int)


# ================================================================
# 3. CONSOLIDATION BY UNIQUE PATIENT (RECORD ID)
# ================================================================
df_patients = df.groupby("Record ID")[["Has_Hypertension", "Has_Diabetes", "Has_Dyslipidemia"]].max().reset_index()
df_patients = df_patients.fillna(0)
total_patients = len(df_patients)


# ================================================================
# 4. CARDIOVASCULAR RISK STRATIFICATION
# ================================================================
def classify_cardiovascular_risk(row):
    if row["Has_Diabetes"] == 1 or (row["Has_Hypertension"] == 1 and row["Has_Dyslipidemia"] == 1):
        return "High Risk (Diabetes or HTN+Dyslipidemia)"
    elif row["Has_Hypertension"] == 1 or row["Has_Dyslipidemia"] == 1:
        return "Moderate Risk (Isolated HTN or Dyslipidemia)"
    else:
        return "Low Risk (No Mapped Factors)"

df_patients["Cardiovascular_Risk_Level"] = df_patients.apply(classify_cardiovascular_risk, axis=1)


# ================================================================
# 4.5. EXPORT DATA TABLES (CSVs FOR THE ARTICLE)
# ================================================================
risk_order = [
    "High Risk (Diabetes or HTN+Dyslipidemia)",
    "Moderate Risk (Isolated HTN or Dyslipidemia)",
    "Low Risk (No Mapped Factors)",
]

risk_counts = df_patients["Cardiovascular_Risk_Level"].value_counts().reindex(risk_order).fillna(0)

risk_table = pd.DataFrame({
    "Cardiovascular Risk Level": risk_counts.index,
    "Unique Patients (N)": risk_counts.values.astype(int),
    "Population Proportion (%)": ((risk_counts.values / total_patients) * 100).round(2),
})
risk_table.to_csv("cardiovascular_risk_stratification_base_en.csv", index=False, encoding="utf-8-sig")

total_diabetes = int(df_patients["Has_Diabetes"].sum())
total_combo = int(
    len(
        df_patients[
            (df_patients["Has_Hypertension"] == 1)
            & (df_patients["Has_Dyslipidemia"] == 1)
            & (df_patients["Has_Diabetes"] == 0)
        ]
    )
)
total_high_risk = int(risk_counts.get("High Risk (Diabetes or HTN+Dyslipidemia)", 0))

if total_high_risk > 0:
    high_risk_profile_table = pd.DataFrame({
        "High-Risk Composition": [
            "Patients with Diabetes (with or without other diseases)",
            "HTN + Dyslipidemia Combination (without Diabetes)",
        ],
        "Patients (N)": [total_diabetes, total_combo],
        "Proportion Within High Risk (%)": [
            round((total_diabetes / total_high_risk) * 100, 2),
            round((total_combo / total_high_risk) * 100, 2),
        ],
    })
    high_risk_profile_table.to_csv("high_risk_profile_base_en.csv", index=False, encoding="utf-8-sig")

print(f"\n{'=' * 80}")
print("DATA TABLES EXPORTED SUCCESSFULLY:")
print("- 'cardiovascular_risk_stratification_base_en.csv' (risk chart data)")
if total_high_risk > 0:
    print("- 'high_risk_profile_base_en.csv' (high-risk breakdown)")
print(f"{'=' * 80}")


# ================================================================
# 5. VISUALIZATION: CARDIOVASCULAR RISK PANEL
# ================================================================
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))

risk_palette = ["#c0392b", "#f39c12", "#2ecc71"]
ax = sns.barplot(x=risk_counts.values, y=risk_counts.index, palette=risk_palette)

plt.title(
    f"Cardiovascular Risk Stratification of the Population\n(Total analyzed base: {total_patients} unique patients)",
    fontsize=16,
    fontweight="bold",
    pad=20,
)
plt.xlabel("Number of Unique Patients", fontsize=13, fontweight="bold")
plt.ylabel("")

for p in ax.patches:
    qty = int(p.get_width())
    pct = (qty / total_patients) * 100
    ax.annotate(
        f" {qty} patients ({pct:.1f}%)",
        (qty, p.get_y() + p.get_height() / 2.0),
        ha="left",
        va="center",
        xytext=(5, 0),
        textcoords="offset points",
        fontweight="bold",
        fontsize=12,
        color="#2c3e50",
    )

sns.despine(left=True, bottom=True)
plt.xlim(0, max(risk_counts.values) * 1.25)
plt.tight_layout()
plt.show()


# ================================================================
# 6. CLINICAL CARE REPORT IN CONSOLE
# ================================================================
print(f"\n{'=' * 90}")
print("POPULATION DIAGNOSIS: CARDIOVASCULAR RISK")
print(f"{'=' * 90}")

high_risk = int(risk_counts.get("High Risk (Diabetes or HTN+Dyslipidemia)", 0))
pct_high = (high_risk / total_patients) * 100 if total_patients > 0 else 0

moderate_risk = int(risk_counts.get("Moderate Risk (Isolated HTN or Dyslipidemia)", 0))
pct_moderate = (moderate_risk / total_patients) * 100 if total_patients > 0 else 0

print(f" -> High Cardiovascular Risk: {high_risk} patients ({pct_high:.1f}%)")
print(f" -> Moderate Risk: {moderate_risk} patients ({pct_moderate:.1f}%)")

print(f"\n[ HIGH-RISK PROFILE ]")
if high_risk > 0:
    print(" Among high-risk patients, we have:")
    print(f"  * {total_diabetes} patients carrying a diagnosis of Diabetes.")
    print(f"  * {total_combo} patients who are not diabetic, but have simultaneous Hypertension + Dyslipidemia.")
else:
    print(" No patient was classified as High Risk.")
print(f"{'=' * 90}\n")